# Hurricane Modeling Notebook

Explores real hurricane track data, trains a PyTorch model for 24h rapid intensification, and saves an artifact.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parents[2]
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from hurricane.model import FEATURE_NAMES, save_model_bundle
from hurricane.scripts.train_model import (
    load_raw_dataset,
    prepare_training_dataframe,
    split_dataset,
    train_model,
)

In [ ]:
raw_path = ROOT / 'src' / 'hurricane' / 'data' / 'raw' / 'hurricane_tracks_merged.csv'
if not raw_path.exists():
    raise FileNotFoundError(
        f'{raw_path} is missing. Run: python src/hurricane/scripts/download_data.py'
    )

raw_df = load_raw_dataset(raw_path)
train_df = prepare_training_dataframe(raw_df)
print(f'Rows: {len(train_df)}')
train_df.head()

In [ ]:
target_rate = train_df['target'].mean()
print(f'RI-positive class rate: {target_rate:.4f}')

axes = train_df[FEATURE_NAMES].hist(figsize=(10, 6), bins=30)
plt.suptitle('Feature Distributions')
plt.tight_layout()

In [ ]:
x_train, y_train, x_val, y_val = split_dataset(train_df, seed=42)
model, feature_mean, feature_std, val_accuracy = train_model(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    epochs=150,
    batch_size=512,
    learning_rate=1e-3,
    seed=42,
)
print(f'Validation accuracy: {val_accuracy:.4f}')

In [ ]:
model_path = ROOT / 'src' / 'hurricane' / 'model' / 'hurricane_model.pt'
model_path.parent.mkdir(parents=True, exist_ok=True)

save_model_bundle(
    path=model_path,
    model=model,
    feature_mean=feature_mean,
    feature_std=feature_std,
    model_version='0.2.0-notebook',
    val_accuracy=val_accuracy,
    dataset_rows=int(len(train_df)),
)
print(f'Saved model artifact to: {model_path}')